# Quickstart: nuclei in 3D

Runs the 2D segmentation of `1_quickstart_2d.ipynb`, then finds the closed envelope of each nucleus around its 2D
outline. Besides the 2D files, `OUT_DIR` receives per stack

- `<name>_labels_3d.tif`: 3D label stack at the input resolution, labels as in the 2D table
- `<name>_nuclei_3d.csv`: volume, centroid, height and z extent per nucleus
- `<name>_sections_3d.png`: outlines on every slice, to check the result

Data and input are as in the 2D quickstart. How the 3D step works: `4_method_3d.ipynb`.

In [ ]:
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

import nucseg
from nucseg.surfaces import plot_3d

In [ ]:
DATA_DIR = Path("data")      # folder with your TIFF z-stacks
OUT_DIR = Path("results")
CHANNEL = None               # channel of multichannel files (0-based); None for single-channel files
UM_PER_PIXEL = None          # pixel size in µm; None reads it from the ImageJ/OME metadata

paths = nucseg.find_stacks([DATA_DIR]) if DATA_DIR.exists() else []
if not paths:
    print(f"No TIFFs in {DATA_DIR}/, using a synthetic demo stack (not a substitute for real images).")
    paths = [nucseg.write_synthetic_stack(Path("example_data") / "synthetic.tif")]
# paths = paths[:1]          # try one stack first (about a minute each)
[p.name for p in paths]

## Parameters

As in the 2D quickstart, plus `z_step_um`, the slice distance: read from the file (ImageJ `spacing`, OME
`PhysicalSizeZ`) unless set here. The stage step often differs from the true distance in the sample (refractive-index
mismatch) and volumes scale with it; see `4_method_3d.ipynb`. Other 3D parameters: `help(nucseg.Parameters3D)`.

In [ ]:
parameters = nucseg.Parameters(
    bin_factor=4,          # pixels averaged per axis, for speed; too coarse loses thin rings
    window=3,              # slices averaged per projection, about 1 µm
    min_area_um2=25.0,     # size of your nuclei
    max_area_um2=800.0,
    min_width_um=5.0,
    z_first=None,          # slices for the 2D outlines (inclusive); None = first / last
    z_last=None,
    border_frame=True,     # also nuclei cut by the image edge, marked border=True (default False)
)
parameters_3d = nucseg.Parameters3D(
    z_step_um=None,        # slice distance in µm; None reads it from the file
)

## Segmentation

About 1 min per 2048 x 2048 x 20 stack (Apple M4 Pro). Check the outlines on every slice and the 2D overview and
gallery in `OUT_DIR`; if nuclei are missing, split or merged, see the table *Tuning* at the end of
`2_method_2d.ipynb`.

In [ ]:
tables, results = [], {}
for path in paths:
    stack = nucseg.read_stack(path, channel=CHANNEL, um_per_pixel=UM_PER_PIXEL)
    z_step = "unknown" if stack.z_step_um is None else f"{stack.z_step_um:.3f} µm"
    print(f"{stack.name}: {stack.data.shape[0]} slices of {stack.data.shape[1]} x {stack.data.shape[2]} px, "
          f"{stack.um_per_pixel:.4f} µm/px, z step {z_step}; segmenting ...")
    segmentation = nucseg.segment_2d(stack, parameters)
    segmentation_3d = nucseg.segment_3d(stack, segmentation, parameters_3d)
    nucseg.save_results(OUT_DIR, stack, segmentation, segmentation_3d)
    results[stack.name] = segmentation_3d
    tables.append(pd.DataFrame(segmentation_3d.table()).assign(image=stack.name))
    print(f"  {len(segmentation.nuclei)} nuclei in 2D, "
          f"{sum(n.found for n in segmentation_3d.nuclei)} in 3D")
    display(Image(OUT_DIR / f"{stack.name}_sections_3d.png"))

## Nuclei

One row per 2D nucleus; `found = False` if no closed envelope was found. If `top_cap` or `bottom_cap` is set, the
nucleus reaches the first or last slice and is closed artificially, so its height and volume are lower bounds (for
`border = True`, they describe the visible part).

In [ ]:
nuclei = pd.concat(tables, ignore_index=True).set_index(["image", "label"])
nuclei

## 3D view

Rotate with the mouse; `z_exaggeration` stretches z for viewing.

In [ ]:
name = next(iter(results))
fig = plot_3d(results[name], z_exaggeration=2.0)
fig.update_layout(height=700, title=name)
fig.write_html(OUT_DIR / f"{name}_3d.html", include_plotlyjs=True)
fig.show()